[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/contracts/blob/main/2-3-classify-train_models_seq.ipynb)

# Train and evaluate classifier

In [ ]:
!pip install -q -U datasets transformers evaluate accelerate

In [ ]:
import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path
import evaluate
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.nn.functional import softmax
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

In [ ]:
clause_type = 'modification' # 'modification' | 'arbitration' | 'anti-scraping' (the clause types in the combined annotations)

# Train on the target sentence only (False) or on the target sentence with its context (True):
#   USE_CONTEXT = False -> input '<sentence>'
#   USE_CONTEXT = True  -> input '<previous> [TGT] <sentence> [TGT] <next>'
USE_CONTEXT = False
CONTEXT_WINDOW = 2   # sentences of context on each side when USE_CONTEXT = True

# judge-annotated data: claude_annotations + the threshold sample judged in 2-6-classify_evaluate.ipynb
ANNOTATION_DIR = Path('annotations/claude_annotations_combined')

# 2-4-classify-apply_models_seq.ipynb loads the model from this directory and reads the settings above
# from classifier_config.json in it
output_dir = f"./models/{clause_type}_best_model_modernbert{'' if USE_CONTEXT else '_sentence'}"

In [12]:

processed_data_dir = Path('processed_data/tous')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')

In [ ]:
annotation_file = ANNOTATION_DIR / f'{clause_type}_annotations.csv'
if not annotation_file.exists():
    raise FileNotFoundError(f'{annotation_file} not found; available: {sorted(p.name for p in ANNOTATION_DIR.glob("*_annotations.csv"))}')
df_annotations = pd.read_csv(annotation_file)
df_annotations.shape

In [ ]:
# Previous_sents / Next_sents were written to CSV as the repr of a Python list
# (see get_surrounding_sentences in 2-1-classify-sample_annotate.ipynb) -- parse them back
for col in ['Previous_sents', 'Next_sents']:
    df_annotations[col] = df_annotations[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
    )
df_annotations['Target_sentence'] = df_annotations['Target_sentence'].fillna('')

# the label column is picked up by prefix ('label_claude' in the combined annotations)
label_col = [c for c in df_annotations.columns if c.startswith('label_')][0]
df_annotations = df_annotations.rename(columns={label_col: 'labels'})
df_annotations['labels'] = df_annotations['labels'].astype(int)

def build_text(row, max_context_sents=CONTEXT_WINDOW):
    """With context: the sentences surrounding the target sentence, with the target wrapped in
    [TGT] ... [TGT] so the model can tell which sentence the label applies to. Without: the sentence."""
    if not USE_CONTEXT:
        return row['Target_sentence']
    prev_txt = ' '.join(row['Previous_sents'][-max_context_sents:])
    next_txt = ' '.join(row['Next_sents'][:max_context_sents])
    return f"{prev_txt} [TGT] {row['Target_sentence']} [TGT] {next_txt}".strip()

df_annotations['text'] = df_annotations.apply(build_text, axis=1)

# The same sentence often occurs in several contract versions (and occasionally with different labels).
# One label per sentence: the majority label; on a tie, the label of the 2-6 threshold sample (newer judge).
source_rank = df_annotations.get('source', pd.Series('', index=df_annotations.index)).eq('2-6_threshold_sample')
df_annotations = df_annotations.assign(_rank=~source_rank).sort_values('_rank', kind='stable').drop(columns='_rank')
votes = df_annotations.groupby('Target_sentence')['labels'].agg(['mean', 'first', 'nunique'])
sentence_label = np.where(votes['mean'] > 0.5, 1, np.where(votes['mean'] < 0.5, 0, votes['first']))
sentence_label = pd.Series(sentence_label, index=votes.index)
print(f"{len(df_annotations)} annotations of {len(votes)} distinct sentences; "
      f"{(votes['nunique'] > 1).sum()} sentences with conflicting labels resolved by majority")
df_annotations['labels'] = df_annotations['Target_sentence'].map(sentence_label).astype(int)

# Without context, repeated sentences are identical examples: keep one. With context, they can differ
# in their context, so all are kept (the split below keeps all copies of a sentence on the same side).
df_annotations = df_annotations.drop_duplicates('text')
df_annotations['labels'].value_counts()

## Train the model

**Model: [`answerdotai/ModernBERT-base`](https://huggingface.co/answerdotai/ModernBERT-base).** A drop-in `AutoModelForSequenceClassification` checkpoint that trains efficiently on an A100 (needs `transformers>=4.48`, already pinned in `requirements.txt`; upgrade with `!pip install -q -U "transformers>=4.48" accelerate` on a fresh Colab runtime). Its 8,192-token window matters when training with context: target sentence + surrounding sentences can exceed the 512-token limit of classic BERT/DistilBERT.

**Input.** With `USE_CONTEXT = False` (default) the model sees only the target sentence. With `USE_CONTEXT = True` it sees `CONTEXT_WINDOW` sentences on each side, with the target wrapped in `[TGT] ... [TGT]` so the model can tell *which* sentence the label applies to.

**Split.** 15% of the *distinct sentences* go to the test set (stratified by label), so a sentence that occurs in several contract versions is never in both the training and the test set.

The labels are imbalanced, so training uses class-weighted cross-entropy, and F1 (not accuracy) drives model selection and early stopping.

In [ ]:
checkpoint = "answerdotai/ModernBERT-base"  # 8k-token context encoder, trains well on an A100
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if USE_CONTEXT:
    tokenizer.add_special_tokens({'additional_special_tokens': ['[TGT]']})

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

In [ ]:
# split on distinct sentences, so copies of a sentence never end up on both sides
train_sents, test_sents = train_test_split(
    sentence_label.index, test_size=0.15, random_state=42, stratify=sentence_label.values
)
train_df = df_annotations[df_annotations['Target_sentence'].isin(set(train_sents))]
test_df = df_annotations[df_annotations['Target_sentence'].isin(set(test_sents))]
print(f"train: {len(train_df)} examples {train_df['labels'].value_counts().to_dict()} | "
      f"test: {len(test_df)} examples {test_df['labels'].value_counts().to_dict()}")

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True)),
    'test':  Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True)),
})

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(device)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
if USE_CONTEXT:
    model.resize_token_embeddings(len(tokenizer))  # account for the [TGT] token added above
model.to(device)

class_weights = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=train_df['labels'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print('class weights (neg, pos):', class_weights)

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy -- positives are only ~6% of
    the data, so an unweighted loss would just learn to always predict 'no'."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


metric = evaluate.combine(["accuracy", "precision", "recall", "f1"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# output_dir is set in the settings cell at the top
import inspect
# 10% linear warmup: transformers 4.x calls this warmup_ratio, 5.x takes the ratio as warmup_steps
warmup = ({'warmup_ratio': 0.1} if 'warmup_ratio' in inspect.signature(TrainingArguments.__init__).parameters
          else {'warmup_steps': 0.1})

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=15,
    per_device_train_batch_size=32,  # A100 40GB+ handles this easily for ModernBERT-base; raise it if you have headroom
    per_device_eval_batch_size=64,
    learning_rate=8e-5,
    **warmup,
    weight_decay=0.01,
    bf16=torch.cuda.is_available(),  # A100 supports bf16 natively; ignored off-GPU
    logging_steps=10,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

eval_metrics = trainer.evaluate()
print(eval_metrics)

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# settings 2-4-classify-apply_models_seq.ipynb needs to build the same input when applying the model
classifier_config = {
    'clause_type': clause_type, 'use_context': USE_CONTEXT, 'context_window': CONTEXT_WINDOW if USE_CONTEXT else 0,
    'annotation_file': str(annotation_file), 'checkpoint': checkpoint,
    'n_train': len(train_df), 'n_test': len(test_df),
    'eval': {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))},
}
with open(Path(output_dir) / 'classifier_config.json', 'w') as f:
    json.dump(classifier_config, f, indent=2)
print(f"Best model saved to: {output_dir}")

## Fin